# 03 · Baseline, read the errors, iterate, freeze

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egumasa/lda2-final-template/blob/main/notebooks/03_prompt.ipynb)

Write the plainest prompt that could work, then improve it for reasons you can state.

```
  01_build_pool_<track>  →  02_sample  →▶ 03_annotate  →  04_prompt  →  05_report
```

| | |
|---|---|
| **Reads** | `data/gold/<track>_<group>_gold.json` (from 02) · the pool (from 01) |
| **Writes** | `outputs/<track>_<group>_predictions.json` · `..._rounds.json` |

---

Everything from here on is measured against **your** gold set, not the corpus's labels. That is the point of the last two notebooks.

> **Free-tier pacing.** The backend waits a few seconds between calls and retries on rate-limit errors, so a full run takes minutes and may print `(rate limited - waiting Ns then retrying)`. That is normal. Keep `N_PER_CLASS` small (2) while you iterate — then do **one** final run at full size.

## Setup — run this first

This cell mounts your Google Drive and finds your group's shared folder, `lda2-final-template`. Everything the project produces — the pool, the gold set, your prompts, the outputs — is an ordinary file in there, which is what makes it survive the runtime resetting *and* lets the rest of your group see it.

**One member sets the folder up once:**

1. That member runs the `git clone` line this cell prints if the folder is missing, which puts it in their own Drive.
2. They share it with the group (right-click ▸ *Share*), with edit access.
3. Everyone else opens *Shared with me*, right-clicks the folder, and chooses **Add shortcut to Drive** ▸ *My Drive*.

Keep that shortcut's name exactly `lda2-final-template`. It is what makes the same path work for all of you — if Drive renames it to `lda2-final-template (1)`, this cell will not find it.

From then on, open notebooks from the folder itself (*File ▸ Open notebook ▸ Drive*) rather than from the GitHub badge, so you are working on your group's copy and not a fresh one.

In [ ]:
# ------------------------------------------------------------------
# SETUP — run me first. You are not expected to read it.
# ------------------------------------------------------------------
# This cell is plumbing, and it is the only cell in the project that is.
# It finds your group's shared folder in Google Drive, because everything
# this project keeps goes in there: a Colab runtime is wiped when it resets,
# and nobody else in your group can see inside it. Then it makes the
# project's own code importable. Run it and move on; nothing below asks you
# to have understood it.

FOLDER = "lda2-final-template"     # the shared folder, in every member's Drive

import os, sys

PROJECT = ".."                              # running locally: it is just above us

try:
    from google.colab import drive           # only exists inside Colab
except ImportError:
    pass
else:
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/" + FOLDER
    if not os.path.isdir(PROJECT):
        raise RuntimeError(
            "Could not find " + PROJECT + "\n\n"
            "Setting the folder up for your group? Run this in a new cell:\n"
            "  !git clone https://github.com/egumasa/lda2-final-template.git "
            + PROJECT + "\n"
            "then share the folder with the rest of your group.\n\n"
            "Someone else already did? Open Drive, find the folder under "
            "'Shared with me', right-click it, and choose 'Add shortcut to "
            "Drive'. Keep the name exactly " + FOLDER + ".")
    # Work inside the project folder, where the notebooks live.
    os.makedirs(PROJECT + "/notebooks", exist_ok=True)
    os.chdir(PROJECT + "/notebooks")

# scripts/ and config.py, by their real paths - so they are found from wherever
# this notebook happens to be working.
sys.path.append(PROJECT)
sys.path.append(PROJECT + "/scripts")

# Re-read config.yaml every time this cell runs. Without the reload, Python
# hands back the settings it read the FIRST time, and editing config.yaml
# would appear to do nothing until you restarted the runtime.
import importlib
import config
importlib.reload(config)

# Named one by one rather than with `import *`, so that every name a cell
# below uses can be traced back to the file it came from — config.yaml for
# these, scripts/ for the rest.
from config import (TRACK, GROUP, RUN, SEED, N_PER_CLASS, MEMBERS,
                    LABELS_ORDER, ROOT, OUT_DIR, POOL_PATH, DEMO_POOL_PATH,
                    SAMPLE_PATH, GOLD_PATH, PRED_PATH, ROUNDS_PATH,
                    PROMPT_FILE, SHEET_PATH, TRIAGE_PATH, describe)

# Files in, files out, and the connection to the model: all plumbing, all
# imported. Asking the model and scoring the answers is what this notebook
# is FOR, so that code is in the notebook, two cells below.
from pipeline import (load_gold, load_prompt, save_json,
                      save_predictions, load_predictions, setup)

setup()                     # connect to the model and say which backend we got

describe()                  # what this notebook is working on


> **Check the backend line it just printed.** You want:
>
> ```
> LLM backend: Gemini API (gemini-3.1-flash-lite, temperature=0, seed=42)
> ```
>
> If it says **Colab Gemini** instead, no API key was found — put yours in the Colab Secrets panel (the 🔑 icon in the left sidebar) as `GEMINI_API_KEY` and re-run. The keyless backend has no temperature or seed, so the same prompt can give different answers and your numbers will not be reproducible. It must not be your final run.

> **Everything above comes from `config.yaml`** — one small file at the top of the repo, which you edit once as a group, and the only file in the plumbing you touch. That is deliberate: the seed that drew your sample has to be the seed you report, and five copies of a number in five notebooks is five chances for them to disagree. Your settings are also the filenames — `track: cars50`, `group: kimura`, `run: v1` means this notebook reads and writes `cars50_kimura_v1_...`. If the line it just printed is not your track, your group and your seed, fix `config.yaml` and re-run this cell.

In [ ]:
# ══ STEP 1 · Load your gold set and your pool ═════════════════════════════
# Goal      : the answers you score against, and the spare items few-shot draws from.
# Available : load_gold(GOLD_PATH)  ·  load_gold(POOL_PATH)
#             label_set(items)  ->  the sorted list of labels present
#             GOLD_PATH · POOL_PATH   (from config.yaml)
# Source    : scripts/pipeline.py · load_gold
# Pointer   : Day 3 setup — the same call, twice.
# Produce   : gold · pool · LABELS      ← later cells use these names
# Note      : LABELS comes from your GOLD set, not the pool. If a label
#             never survived adjudication, it is not in your study.

# ✏️ your code here — fill in each ____

gold = load_gold(____)          # GOLD_PATH — your own labels, from notebook 03
pool = load_gold(____)          # POOL_PATH — the spares few-shot draws from
LABELS = label_set(____)        # gold, not pool

print(len(gold), "gold ·", len(pool), "pool ·", LABELS)


### The code that does it — read it, then run it

These three functions are the measurement itself. `run_prompt` is the loop that asks the model once per item; `extract_label` is the guess about what its reply *meant*; `build_fewshot` is how examples get in front of it. Every number in your report comes out of these, so read them before you trust them.

It is read straight out of `scripts/` when this notebook is generated, so it is not a simplified copy: it is the code that runs. Two things to look for as you read:

- **`extract_label` is doing more than it looks.** The model answers in prose; something has to decide that *"This looks like Move 2 to me"* is `Move 2`. It searches for label names, keeps the longest match, and falls back to `"??"`. Every `??` in your run is a reply this function could not read — and if there are many, that is a finding about your prompt, not a bug.
- **`build_fewshot` skips anything in your gold set**, matching by text rather than by id (sampling renumbered the ids). Without that, you would be testing the model on answers you had just shown it.

Run the cell to define these, then use them in the step below.

In [ ]:
# Read straight out of scripts/pipeline.py — this IS the code that runs.

import random, re

# The two helpers these lean on: `label_set` you have met, and
# `_default_backend` is the connection `setup()` opened above.
from pipeline import label_set, _default_backend

def extract_label(reply, labels):
    """Figure out which of the known labels the model's reply is pointing at.

    Returns "??" when we cannot find any known label in the reply.
    """
    reply_text = str(reply).strip()
    reply_lowercased = reply_text.lower()

    # Step 1: collect every known label whose name appears in the reply.
    labels_found = []
    for label in labels:
        if label.lower() in reply_lowercased:
            labels_found.append(label)

    # Step 2: if we found one or more, keep the longest (most specific) one.
    if len(labels_found) > 0:
        longest_label = labels_found[0]
        for label in labels_found:
            if len(label) > len(longest_label):
                longest_label = label
        return longest_label

    # Step 3: special case for "Move 1/2/3" labels - look for a bare digit.
    has_move_labels = False
    for label in labels:
        if label.lower().startswith("move "):
            has_move_labels = True
    if has_move_labels:
        match = re.search(r"\b([1-9])\b", reply_text)
        if match is not None:
            candidate = "Move " + match.group(1)
            if candidate in labels:
                return candidate

    # Step 4: nothing matched.
    return "??"

def run_prompt(prompt, gold, labels=None, generate_text=None):
    """Ask the model to label every item, and collect the predicted labels.

    Same call as Day 3: run_prompt(PROMPT, gold). The two optional arguments are
    worked out for you - `labels` from the gold set, and the model connection from
    the Setup cell - so you only pass them if you want something different.
    """
    if labels is None:
        labels = label_set(gold)
    if generate_text is None:
        generate_text = _default_backend()

    # A prompt that asks for {context} on a track whose items have none would quietly
    # send the model an empty passage, once per item, and report a number as if it had
    # tested something. Say so instead.
    if "{context}" in prompt and not any(item.get("context") for item in gold):
        print("WARNING: this prompt uses {context}, but none of these items carry one. "
              "Only the rhetorical-move tracks (cars50, raamove) do. The model is about "
              "to be shown an empty passage " + str(len(gold)) + " times.")

    predictions = []
    total = len(gold)
    position = 0
    for item in gold:
        position = position + 1
        # Put this item's sentence into the prompt where {text} is - and its passage
        # where {context} is, on the tracks that carry one. A prompt that does not
        # mention {context} simply ignores it.
        filled_prompt = prompt.format(text=item["text"],
                                      context=item.get("context", ""))
        reply = generate_text(filled_prompt)
        predicted_label = extract_label(reply, labels)
        predictions.append(predicted_label)
        # Print a small progress note every 10 items.
        if position % 10 == 0:
            print("  ...", position, "/", total, "done")

    # Count how many replies we could not turn into a valid label.
    number_unparseable = 0
    for label in predictions:
        if label == "??":
            number_unparseable = number_unparseable + 1
    print("Got", len(predictions), "predictions (", number_unparseable, "could not be parsed).")
    return predictions

def build_fewshot(base_prompt, pool, gold, labels=None, shots_per_class=1, seed=42):
    """Put a few labeled examples (taken from the pool) in front of the prompt.

    We NEVER use an item that is in the gold set as an example, otherwise we
    would be showing the model the very answers we are testing it on. Items are
    matched by their TEXT, not their id, because sampling renumbers the ids.
    """
    if labels is None:
        labels = label_set(gold)

    # Step 1: collect the texts that are already in the gold set.
    gold_texts = []
    for item in gold:
        gold_texts.append(item["text"])

    # Step 2: group the remaining pool items by label (skipping any gold items).
    examples_by_label = {}
    for item in pool:
        if item["text"] in gold_texts:
            continue
        label = item["label"]
        if label not in examples_by_label:
            examples_by_label[label] = []
        examples_by_label[label].append(item)

    # Step 3: for each label, shuffle and take a few examples.
    random_generator = random.Random(seed)
    lines = ["Here are labeled examples:"]
    labels_with_no_examples = []
    for label in labels:
        if label in examples_by_label:
            examples = examples_by_label[label]
        else:
            examples = []
        random_generator.shuffle(examples)
        chosen_examples = examples[:shots_per_class]
        if len(chosen_examples) < shots_per_class:
            labels_with_no_examples.append(label)
        for item in chosen_examples:
            lines.append("Sentence: " + item["text"] + "\nLabel: " + label)

    # Step 4: if the pool could not supply enough spare examples, say so loudly -
    # a few-shot prompt missing whole labels is not the prompt you think it is.
    if len(labels_with_no_examples) > 0:
        print("WARNING: not enough spare pool items for", shots_per_class,
              "example(s) of:", ", ".join(labels_with_no_examples))
        print("         Those labels get fewer examples (or none). This usually means")
        print("         POOL_PATH points at a small DEMO file rather than a full pool.")

    # Step 5: glue the example block in front of the base prompt.
    example_block = "\n\n".join(lines)
    return example_block + "\n\nNow classify this one.\n\n" + base_prompt

### The code that does it — read it, then run it

And this is the scoring. `evaluate` prints per-class precision/recall/F1, Cohen's κ, and the confusion matrix — and **returns the macro-F1 as a number**, which is what lets you collect one per round. `show_errors` gives you the items it got wrong, and you will use it in **every round**, not just at the end.

It is read straight out of `scripts/` when this notebook is generated, so it is not a simplified copy: it is the code that runs. Two things to look for as you read:

- **Macro-F1** is the plain average of the per-class F1 scores — every class counts the same, however rare. That is why a balanced sample and a macro average go together.
- **`ordered=True` adds a *weighted* κ**, which counts a near miss (Low→Mid) as a smaller error than a far one (Low→High). Use it only if your labels sit on a scale, and pass `labels=LABELS_ORDER` so it knows what that scale is.
- **`show_errors` is the one you will actually iterate on.** F1 tells you *whether* a round helped; only the errors tell you *what to change next*.

Run the cell to define these, then use them in the step below.

In [ ]:
# Read straight out of scripts/metrics.py — this IS the code that runs.

import pandas as pd
from sklearn.metrics import (classification_report, confusion_matrix,
                             cohen_kappa_score, f1_score)
from pipeline import plot_confusion_matrix

def evaluate(gold, predictions, ordered=False, labels=None, title="Confusion matrix"):
    """Score predictions against gold: per-class P/R/F1 + macro, Cohen's kappa, and a
    confusion-matrix heatmap. Returns the macro-F1 as a number.

    ordered=True adds QUADRATIC WEIGHTED kappa — use it only when the labels sit on a
    scale (A1 < A2 < ... < C2), so that a near miss counts as a smaller error than a
    far one. For unordered categories, plain kappa is the one to report.

    IMPORTANT for ordered=True: the scale is taken from `labels`, in the order given.
    Left off, `labels` is read off the gold set and sorted ALPHABETICALLY — which is
    correct for A1..C2 and Move 1..3, but wrong for something like Low/Mid/High
    (alphabetical puts High first). If your labels are ordered and not alphabetical,
    pass them yourself: evaluate(gold, pred, ordered=True, labels=LABELS_ORDER).
    """
    # --- Compatibility with the older 4-positional call form -----------------------
    # An earlier version of this file took evaluate(gold, predictions, labels, title).
    # If we were called that way, argument 3 is a list of labels rather than a
    # true/false flag. Rather than fail with a confusing error - or worse, silently
    # treat a non-empty list as "ordered=True" - detect it and shuffle the arguments.
    if isinstance(ordered, (list, tuple)):
        print("NOTE: old call form evaluate(gold, pred, labels, title) — treating "
              "argument 3 as labels. The current form is "
              "evaluate(gold, pred, ordered=..., labels=...).")
        if isinstance(labels, str):
            title = labels
        labels = list(ordered)
        ordered = False

    ### Step 1: line the two label lists up, gold first ###
    y_true = []                          # the correct labels, from the gold set
    for item in gold:
        y_true.append(item["label"])
    y_pred = predictions                 # the model's labels, in the same order

    if labels is None:
        labels = label_set(gold)

    ### Step 2: per-class precision / recall / F1, as a text table ###
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

    ### Step 3: one overall number — agreement corrected for chance ###
    # Only meaningful if there is more than one label to be right or wrong about.
    if len(set(labels)) < 2:
        print("Cohen's kappa            undefined (only one label present)")
    else:
        print(f"Cohen's kappa            {cohen_kappa_score(y_true, y_pred):.3f}")
        if ordered:                      # only when the labels sit on a scale
            weighted = cohen_kappa_score(y_true, y_pred, labels=labels,
                                         weights="quadratic")   # near misses hurt less
            print(f"Cohen's kappa (weighted) {weighted:.3f}   <- labels are ordered")
            # Say WHICH order we used, so a wrong one is visible rather than silent.
            print("  scale order used:", " < ".join(labels))

    ### Step 4: draw the same information as a picture ###
    matrix = confusion_matrix(y_true, y_pred, labels=labels)
    plot_confusion_matrix(matrix, labels, title)

    ### Step 5: one number to carry from round to round ###
    macro_f1 = f1_score(y_true, y_pred, labels=labels,
                        average="macro", zero_division=0)
    return macro_f1

def show_errors(gold, predictions):
    """The items the model got wrong, as a table you can read and argue about."""
    rows = []
    for item, predicted in zip(gold, predictions):
        if item["label"] != predicted:
            row = {
                "id": item["id"],
                "gold": item["label"],
                "pred": predicted,
                "text": item["text"],
            }
            rows.append(row)
    print(f"{len(rows)} of {len(gold)} wrong.")
    return pd.DataFrame(rows)             # a table, so Colab displays it nicely

## Step 2 — The baseline (round 0)

A number to beat. Write the plainest prompt that states the task and the label set, run it, score it. **Resist the urge to make it good** — the point of a baseline is that later rounds have something to be measured against, and a baseline you already tuned tells you nothing about whether tuning helped.

Your prompt lives in `prompts/<track>.txt` and must contain `{text}`, where each item gets slotted in. Edit the **file**, not a string in this notebook — that is what makes each version savable and comparable, and it is the reproducibility habit from S10.

In Colab you can write the file straight from a cell:

```python
%%writefile ../prompts/raamove_v0.txt
Classify the rhetorical move of the sentence. Answer with the move name only.
...

Sentence: {text}
```

In [ ]:
# ══ STEP 2 · Baseline prompt (round 0) — and read what it got wrong ═══════
# Goal      : get one honest number to beat, and the errors that tell you what to fix.
# Available : load_prompt(PROMPT_FILE)  ->  PROMPT
#             run_prompt(PROMPT, gold)  ->  predictions
#             evaluate(gold, predictions, ordered=..., labels=LABELS_ORDER)  ->  macro-F1
#             show_errors(gold, predictions)  ->  the items it got wrong
#             PROMPT_FILE   (config.yaml: prompts/<track>.txt)
# Source    : the two cells above · scripts/pipeline.py · scripts/metrics.py
# Pointer   : Day 3 Part A — the same two lines, plus the error table.
# Produce   : f1_by_round · pred0      ← later cells use these names
# Note      : `f1_by_round` is your prompt-iteration table — one entry per
#             round, keyed by the round's name. Notebook 04 reads it back
#             from a file and prints it into the report, so the keys are
#             what your reader sees: name them so they mean something.
# Note      : do not skip the last line. The errors are the ONLY thing that
#             tells you what to change; F1 only tells you afterwards
#             whether the change worked.
# Careful   : keep N_PER_CLASS small for this. Full size is minutes of
#             pure waiting per round on the free tier.

# ✏️ your code here — fill in each ____

# One entry per round from here on. Notebook 04 turns it into your table.
f1_by_round = {}

PROMPT = load_prompt(____)               # PROMPT_FILE
print(PROMPT)

pred0 = run_prompt(PROMPT, gold)
f1_by_round["round0 baseline"] = evaluate(gold, pred0,
                                          ordered=____,   # True only if your labels are a SCALE
                                          labels=LABELS_ORDER)

# Now look at what it actually got wrong. This is the cell that decides
# your next round.
show_errors(gold, pred0)


## Step 3 — Iterate, driven by the errors

Two or three more rounds. The loop is always the same, and the middle step is the one that matters:

```
run  →  score  →  READ THE ERRORS  →  change ONE thing  →  run again
```

Before you touch the prompt, look at the error table from the round you just ran and ask **what these misses have in common**. There are only a few answers, and each points somewhere different:

| What you see in the errors | What it suggests |
|---|---|
| One class swallows everything | the model has not understood that class's boundary — define it, or show an example of it |
| Two labels traded in both directions | the *distinction* is unclear, to the model and possibly to your coders too |
| Lots of `??` | the model is not answering in the format you asked for — fix the instruction, not the definitions |
| Errors scattered with no pattern | you may be at the ceiling of what the prompt can do; consider whether the items are simply hard |

Then change **one** thing, and say beforehand what you expect it to do. "Added examples" is not a reason; *"Move 2 and Move 3 traded in both directions, so I gave it one example of each"* is. Write it down as you go — reconstructing it afterwards from a stack of F1 numbers is much harder than it sounds, and it is report section 2.

A round that made things **worse** is a result, not a mistake. Keep it in the table. It is often the most informative row you have.

`build_fewshot` draws examples from the pool while avoiding anything in your gold set — otherwise you would be testing the model on answers you had just shown it.

In [ ]:
# ══ STEP 3 · Iterate — 2–3 rounds, each one justified by the errors ═══════
# Goal      : change one thing per round, for a reason you can point at in the error table.
# Available : build_fewshot(PROMPT, pool, gold)  ->  a new prompt with examples
#             run_prompt(...)  ·  evaluate(...)  ·  show_errors(...)   (as in step 2)
# Source    : the cells above · scripts/pipeline.py · build_fewshot
# Pointer   : Day 3 iterations 1–2. build_fewshot replaces typing the examples by hand.
# Produce   : PROMPT (your best one) · f1_by_round      ← later cells use these names
# Note      : save each version as its own prompt file (v0, v1, v2). A
#             prompt you overwrote is a round you cannot report.
# Ask       : did the confusion matrix change SHAPE, or did everything
#             shift a little? Those need different next moves.

# ✏️ your code here — fill in each ____

# ONE round. Copy this whole block for round 2, and again for round 3.

# ── What did you see in the LAST round's errors, and what are you
#    changing because of it? One line each, before you run anything. ──
#
#   I saw   >>> 
#   So I    >>> 
#   I expect>>> 

# Either add examples to the prompt you have …
PROMPT = build_fewshot(PROMPT, pool, gold)
# … or write a new prompt file (see the %%writefile example above) and
# load that instead:
# PROMPT = load_prompt("../prompts/____.txt")

pred1 = run_prompt(PROMPT, gold)
f1_by_round["round1 ____"] = evaluate(gold, pred1,      # name the CHANGE
                                      ordered=____, labels=LABELS_ORDER)

# Read these before you design round 2.
show_errors(gold, pred1)


## Step 4 — Freeze

A hosted model is only *best-effort* reproducible, even at `temperature=0`. So once your best prompt is settled:

1. Raise `n_per_class` in `config.yaml` to full size — and re-run notebooks 02 and 03 if that changes your sample. (If it does, you have more annotating to do. This is why you decide the size **before** you annotate.)
2. Run the model **once**, on your best prompt.
3. `save_json` the predictions to a file.

Every number you report from here on comes out of that file. That is what makes your F1 hold still, and what lets anyone else re-run your analysis on exactly the outputs you saw.

**One person runs this.** It is the run you will be defending.

In [ ]:
# ══ STEP 4 · The final run, frozen to a file ══════════════════════════════
# Goal      : one full-size run on your best prompt, saved so the numbers stop moving.
# Available : run_prompt(PROMPT, gold)  ->  predictions
#             save_predictions(predictions, PRED_PATH)  ·  load_predictions(PRED_PATH)
#             save_json(f1_by_round, ROUNDS_PATH, what="rounds")
# Source    : scripts/pipeline.py · save_predictions, load_predictions, save_json
# Pointer   : Day 2 S6 loaded a frozen file we made; now you make your own.
# Produce   : pred_final      ← later cells use these names
# Freeze    : save, then load it straight back and use THAT from now on.
#             Reading it back is not superstition — it is the check that
#             the file you will report from is the file you think it is.
# Careful   : save f1_by_round too. Notebook 04 needs it, and it is the
#             one thing here that exists only in this session's memory.

# ✏️ your code here — fill in each ____

predictions = run_prompt(____, gold)          # PROMPT — your best one
save_predictions(predictions, PRED_PATH)

# Report from the FILE, not from the variable: this is the check that the
# file you will quote is the file you think it is.
pred_final = load_predictions(PRED_PATH)

# The rounds table exists only in this session's memory until now.
save_json(f1_by_round, ROUNDS_PATH, what="rounds")


---

**Next:** open `05_report.ipynb`. It loads the two files you just wrote and nothing else — so from here on, your numbers cannot move.